# 01 · Data Preparation
**Medical Chatbot MVP** - dataset preparation and validation

This notebook:
1. Loads and inspects raw data
2. Validates structure and coverage
3. Builds documents for RAG indexing
4. Saves processed data
5. Generates evaluation dataset (questions + golden answers)


In [29]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

DATA_DIR = Path('../data')
RAW_DIR = DATA_DIR / 'raw'
KB_DIR = DATA_DIR / 'processed' / 'knowledge_base'
KB_DIR.mkdir(parents=True, exist_ok=True)

print('Setup OK')

Setup OK


In [30]:
with open(RAW_DIR / 'doctors.json', encoding='utf-8') as f:
    doctors = json.load(f)

with open(RAW_DIR / 'specialties.json', encoding='utf-8') as f:
    specialties = json.load(f)

with open(RAW_DIR / 'appointments.json', encoding='utf-8') as f:
    appointments = json.load(f)

df_doctors = pd.DataFrame(doctors)
df_specs = pd.DataFrame(specialties)

print(f'Doctors: {len(doctors)}')
print(f'Specialties: {len(specialties)}')
print(f'Appointments: {len(appointments)}')
print()
df_doctors[['id','name','specialty','years_experience','rating','accepts_nhif']].head(10)

Doctors: 18
Specialties: 12
Appointments: 18



,id,name,specialty,years_experience,rating,accepts_nhif
0,dr_001,д-р Мария Иванова,кардиолог,18,4.9,False
1,dr_002,д-р Георги Петров,кардиолог,9,4.7,True
2,dr_003,д-р Елена Николова,невролог,22,4.9,False
3,dr_004,д-р Стефан Димитров,невролог,11,4.6,True
4,dr_005,д-р Христо Стоянов,ортопед,20,4.8,False
5,dr_006,д-р Ива Маринова,ортопед,8,4.7,True
6,dr_007,д-р Деница Тодорова,дерматолог,13,4.8,False
7,dr_008,д-р Калин Велчев,дерматолог,7,4.6,True
8,dr_009,д-р Радостина Коева,гастроентеролог,17,4.9,False
9,dr_010,д-р Пламен Александров,гастроентеролог,10,4.6,True


In [31]:
required_fields = ['id','name','specialty_id','specialty','bio','expertise',
                    'languages','price_consultation','rating','room']

print('=== Doctor validation ===')
issues = []
for doc in doctors:
    for field in required_fields:
        if field not in doc or doc[field] is None:
            issues.append(f"{doc['id']} — missing field '{field}'")

if issues:
    print('ISSUES FOUND:')
    for i in issues: print(i)
else:
    print('All required fields present')

spec_ids = {s['id'] for s in specialties}
doctor_spec_ids = {d['specialty_id'] for d in doctors}
missing_specs = doctor_spec_ids - spec_ids

if missing_specs:
    print(f'ERROR: Unknown specialty_id values: {missing_specs}')
else:
    print('All specialty_id values are valid')

doctor_ids = {d['id'] for d in doctors}
appt_ids = set(appointments.keys())
no_appt = doctor_ids - appt_ids

if no_appt:
    print(f'WARNING: Doctors without appointment slots: {no_appt}')
else:
    print('All doctors have appointment slots')

=== Doctor validation ===
All required fields present
All specialty_id values are valid
All doctors have appointment slots


## Dataset distribution

In [32]:
# Doctors per specialty
spec_counts = df_doctors['specialty'].value_counts().reset_index()
spec_counts.columns = ['specialty', 'count']

fig = px.bar(spec_counts, x='count', y='specialty', orientation='h',
             title='Doctors per specialty',
             color='count', color_continuous_scale='Blues')
fig.update_layout(showlegend=False, yaxis={'categoryorder':'total ascending'})
fig.show()

print(f"Avg consultation price: {df_doctors['price_consultation'].mean():.0f} EUR")
print(f"Avg rating: {df_doctors['rating'].mean():.2f}")
print(f"Doctors accepting NHIF: {df_doctors['accepts_nhif'].sum()} of {len(df_doctors)}")

Avg consultation price: 93 EUR
Avg rating: 4.77
Doctors accepting NHIF: 7 of 18


In [33]:
# Symptom coverage per specialty
all_symptoms = []
for spec in specialties:
    for sym in spec['symptoms']:
        all_symptoms.append({'specialty': spec['specialty'], 'symptom': sym})

df_sym = pd.DataFrame(all_symptoms)
print(f'Total symptom entries: {len(df_sym)}')
print(f'Unique symptoms: {df_sym["symptom"].nunique()}')
print()
print('Symptoms per specialty:')
print(df_sym.groupby('specialty')['symptom'].count().sort_values(ascending=False).to_string())

Total symptom entries: 150
Unique symptoms: 144

Symptoms per specialty:
specialty
гастроентеролог    16
дерматолог         16
невролог           15
ортопед            15
ендокринолог       13
психиатър          13
кардиолог          12
УНГ лекар          12
пулмолог           10
офталмолог         10
уролог             10
гинеколог           8


In [34]:
# Appointment slot availability
stats = []
for doc_id, data in appointments.items():
    total = len(data['slots'])
    available = sum(1 for s in data['slots'] if s['available'])
    stats.append({
        'id': doc_id,
        'name': data['doctor_name'],
        'total': total,
        'available': available,
        'booked': total - available,
        'availability_pct': round(available / total * 100) if total > 0 else 0
    })

df_appt = pd.DataFrame(stats)
print(f'Total slots: {df_appt["total"].sum()}')
print(f'Available: {df_appt["available"].sum()}')
print(f'Booked: {df_appt["booked"].sum()}')
print(f'Avg occupancy: {100 - df_appt["availability_pct"].mean():.0f}%')
df_appt[['name','total','available','availability_pct']].head(10)

Total slots: 4360
Available: 2835
Booked: 1525
Avg occupancy: 35%


,name,total,available,availability_pct
0,д-р Мария Иванова,221,136,62
1,д-р Георги Петров,136,86,63
2,д-р Елена Николова,153,100,65
3,д-р Стефан Димитров,204,130,64
4,д-р Христо Стоянов,221,157,71
5,д-р Ива Маринова,204,135,66
6,д-р Деница Тодорова,357,230,64
7,д-р Калин Велчев,221,151,68
8,д-р Радостина Коева,357,231,65
9,д-р Пламен Александров,204,129,63


## Build RAG documents

Transform structured JSON data into natural language documents suitable for embedding and retrieval.

In [35]:
def doctor_to_document(doc: dict) -> dict:
    """
    Convert a doctor JSON record into a text document for RAG.
    Returns a dict with 'text' (for embedding) and 'metadata' (for filtering).
    The document text is intentionally kept in Bulgarian — the language of the data.
    """
    nhif_text = 'Приема по НЗОК с направление' if doc.get('accepts_nhif') else 'Само платени прегледи'
    academic = f"{doc['academic_title']} " if doc.get('academic_title') else ''
    expertise = ', '.join(doc.get('expertise', []))
    languages = ', '.join(doc.get('languages', ['български']))

    text = f"""Лекар: {academic}{doc['name']}
Специалност: {doc['specialty']}
Опит: {doc['years_experience']} години
Образование: {doc.get('education', '')}

{doc.get('bio', '')}

Специализира в: {expertise}
Езици: {languages}
Цена на преглед: {doc['price_consultation']} евро
{nhif_text}
Оценка: {doc['rating']}/5.0 ({doc.get('reviews_count', 0)} отзива)
Кабинет: {doc.get('room', 'N/A')}"""

    return {
        'id': f"doctor_{doc['id']}",
        'text': text,
        'metadata': {
            'type': 'doctor',
            'doctor_id': doc['id'],
            'specialty_id': doc['specialty_id'],
            'specialty': doc['specialty'],
            'accepts_nhif': doc.get('accepts_nhif', False),
            'price': doc['price_consultation'],
            'rating': doc['rating'],
        }
    }

doctor_docs = [doctor_to_document(d) for d in doctors]
print(f'Generated {len(doctor_docs)} doctor documents')
print()
print('--- Sample (dr_001) ---')
print(doctor_docs[0]['text'])

Generated 18 doctor documents

--- Sample (dr_001) ---
Лекар: доцент д-р Мария Иванова
Специалност: кардиолог
Опит: 18 години
Образование: Медицински университет — София. Специализация по кардиология — Виена, Австрия.

Доц. д-р Мария Иванова е кардиолог с 18-годишен опит в диагностика и лечение на сърдечно-съдови заболявания. Специализирала е инвазивна кардиология в Medizinische Universität Wien. Извършва над 500 коронарографии годишно. Говори свободно на английски и немски.

Специализира в: исхемична болест на сърцето, сърдечна недостатъчност, аритмии, артериална хипертония
Езици: български, английски, немски
Цена на преглед: 100 евро
Само платени прегледи
Оценка: 4.9/5.0 (312 отзива)
Кабинет: 102


In [36]:
def specialty_to_document(spec: dict) -> dict:
    """
    Convert a specialty record into a document mapping symptoms to specialist.
    This is the core of the routing logic.
    Document text is in Bulgarian to match user queries.
    """
    symptoms_text = ', '.join(spec['symptoms'])
    urgent_text = ', '.join(spec.get('urgent_symptoms', []))
    procedures_text = ', '.join(spec.get('common_procedures', []))

    text = f"""Специалност: {spec['specialty']}
{spec['description']}

При кои симптоми трябва да се запише час при {spec['specialty']}:
{symptoms_text}

СПЕШНИ симптоми — незабавно потърсете помощ (112 или Спешно отделение):
{urgent_text}

Типични изследвания и процедури:
{procedures_text}

Продължителност на преглед: около {spec['avg_consultation_minutes']} минути"""

    return {
        'id': f"specialty_{spec['id']}",
        'text': text,
        'metadata': {
            'type': 'specialty',
            'specialty_id': spec['id'],
            'specialty': spec['specialty'],
        }
    }

specialty_docs = [specialty_to_document(s) for s in specialties]
print(f'Generated {len(specialty_docs)} specialty documents')
print()
print('--- Sample (cardiology) ---')
print(specialty_docs[0]['text'])

Generated 12 specialty documents

--- Sample (cardiology) ---
Специалност: кардиолог
Специалист по сърдечно-съдови заболявания. Диагностицира и лекува проблеми със сърцето, артериите и вените.

При кои симптоми трябва да се запише час при кардиолог:
болка в гърдите, стягане в гърдите, задух при усилие, задух в покой, сърцебиене, неравномерен пулс, замайване, припадък, подуване на краката, умора при минимално усилие, високо кръвно налягане, ниско кръвно налягане

СПЕШНИ симптоми — незабавно потърсете помощ (112 или Спешно отделение):
остра болка в гърдите, болка в гърдите с изпотяване, болка в гърдите с гадене, загуба на съзнание, внезапен задух

Типични изследвания и процедури:
ЕКГ, ехокардиография, холтер мониториране, стрес тест

Продължителност на преглед: около 30 минути


In [37]:
# Load knowledge base .txt files and split into paragraph-level chunks
kb_docs = []
for kb_file in sorted(KB_DIR.glob('*.txt')):
    text = kb_file.read_text(encoding='utf-8')
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

    for i, para in enumerate(paragraphs):
        if len(para) < 50:  # skip very short lines (headers, separators)
            continue
        kb_docs.append({
            'id': f"kb_{kb_file.stem}_{i:03d}",
            'text': para,
            'metadata': {
                'type': 'knowledge_base',
                'source': kb_file.stem,
            }
        })

print(f'Loaded {len(kb_docs)} KB chunks from {len(list(KB_DIR.glob("*.txt")))} files')

# Merge all document types
all_documents = doctor_docs + specialty_docs + kb_docs
print(f'\nTotal RAG documents: {len(all_documents)}')
print(f'Doctors: {len(doctor_docs)}')
print(f'Specialties: {len(specialty_docs)}')
print(f'KB chunks: {len(kb_docs)}')

Loaded 39 KB chunks from 3 files

Total RAG documents: 69
Doctors: 18
Specialties: 12
KB chunks: 39


In [38]:
# Save processed documents to disk
processed_path = DATA_DIR / 'processed' / 'rag_documents.json'
with open(processed_path, 'w', encoding='utf-8') as f:
    json.dump(all_documents, f, ensure_ascii=False, indent=2)

print(f'Saved to: {processed_path}')

# Character length stats per document type
chars = [len(d['text']) for d in all_documents]
df_chars = pd.DataFrame({
    'chars': chars,
    'type': [d['metadata']['type'] for d in all_documents]
})

print(f'\nAvg document length: {sum(chars)/len(chars):.0f} chars')
print(f'Min: {min(chars)} Max: {max(chars)}')
print()
print(df_chars.groupby('type')['chars'].agg(['mean','min','max']).round(0))

Saved to: ..\data\processed\rag_documents.json

Avg document length: 349 chars
Min: 67 Max: 751

                 mean  min  max
type                           
doctor          563.0  507  652
knowledge_base  159.0   67  332
specialty       643.0  517  751


## Generate evaluation dataset

20 questions with golden answers used by `03_rag_evaluation.ipynb` to score retrieval quality.

In [39]:
with open("../data/processed/eval_dataset.json", "r", encoding="utf-8") as f:
    eval_dataset = json.load(f);

eval_path = DATA_DIR / 'processed' / 'eval_dataset.json'
with open(eval_path, 'w', encoding='utf-8') as f:
    json.dump(eval_dataset, f, ensure_ascii=False, indent=2)

df_eval = pd.DataFrame(eval_dataset)
print(f'Evaluation dataset saved: {len(eval_dataset)} questions')
print()
print('By category:')
print(df_eval['category'].value_counts().to_string())
print()
print('By urgency:')
print(df_eval['urgency'].value_counts().to_string())

Evaluation dataset saved: 20 questions

By category:
category
symptom_routing    10
doctor_info         4
center_info         3
urgency             2
ambiguous           1

By urgency:
urgency
low          11
medium        5
high          2
emergency     2


## Summary

In [40]:
print('=' * 50)
print('DATASET SUMMARY')
print('=' * 50)
print(f'Doctors: {len(doctors)}')
print(f'Specialties: {len(specialties)}')
print(f'Appointment slots: {sum(len(v["slots"]) for v in appointments.values())}')
print(f'RAG documents: {len(all_documents)}')
print(f'Eval questions: {len(eval_dataset)}')
print()
print('Files in data/processed/')
for p in sorted((DATA_DIR / 'processed').rglob('*')):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f'{str(p.relative_to(DATA_DIR/"processed")):<40} {size_kb:.1f} KB')
print()

DATASET SUMMARY
Doctors: 18
Specialties: 12
Appointment slots: 4360
RAG documents: 69
Eval questions: 20

Files in data/processed/
eval_dataset.json                        7.8 KB
eval_results.json                        16.1 KB
faiss_index\embeddings_raw.npy           414.1 KB
faiss_index\index.faiss                  414.0 KB
faiss_index\metadata.pkl                 4.9 KB
faiss_index\texts.pkl                    43.5 KB
knowledge_base\center_info.txt           3.1 KB
knowledge_base\faq.txt                   4.2 KB
knowledge_base\services.txt              3.7 KB
rag_documents.json                       55.2 KB

